# 🧳 Solution: Intelligent Travel Assistant

## Learning Objectives
This worked answer to `modules/03_react_agents/exercises/01_intelligent_travel_assistant_agent.ipynb` shows:
1. **Two robust tools** - Weather from OpenWeatherMap and web search with page extraction, both failing soft
2. **Checking tool selection** - Confirming the model picks both tools before building the agent
3. **The agent** - A system prompt and `create_agent` with a model-call limit
4. **Reading the run** - Which tools were called, and how the agent reasoned its way to the answer

## 🏗️ Architecture

```
User Query: "Tell me about Kolkata"
              │
              ▼
┌─────────────────────────────────────────┐
│         Travel Assistant Agent          │
├─────────────────────────────────────────┤
│  System prompt: travel assistant that   │
│  returns attractions + current weather  │
└─────────────────────────────────────────┘
              │
    ┌─────────┴─────────┐
    ▼                   ▼
┌────────────┐    ┌─────────────┐
│ Web Search │    │   Weather   │
│    Tool    │    │    Tool     │
└────────────┘    └─────────────┘
    │                   │
    ▼                   ▼
┌────────────┐    ┌─────────────┐
│ Tavily API │    │ OpenWeather │
└────────────┘    └─────────────┘
```

## Prerequisites
- Try the exercise first
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY`, `TAVILY_API_KEY` and `WEATHER_API_KEY` (from [OpenWeatherMap](https://openweathermap.org/api)) in the project-root `.env`

---
## 📦 Step 1: Environment Setup

Load the keys (the cell only reports whether each is present), create the LLM, and set up the search plumbing.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and check they are present
# ============================================================================
import os
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY, TAVILY_API_KEY and WEATHER_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

for key in ["OPENAI_API_KEY", "TAVILY_API_KEY", "WEATHER_API_KEY"]:
    print(f"{'✅' if os.getenv(key) else '❌ missing'} {key}")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: The travel assistant's reasoning engine
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

In [ ]:
# ============================================================================
# SEARCH PLUMBING: a Tavily search tool and a web-page-to-text converter
# ============================================================================
import requests
from langchain_tavily import TavilySearch
from markitdown import MarkItDown

# Tavily finds pages; basic depth is faster and enough for attraction queries
tavily_search = TavilySearch(max_results=3, search_depth="basic")

# Certain websites won't let you crawl them unless you send a browser-like user agent
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
})
# MarkItDown converts a web page into readable text
md_converter = MarkItDown(requests_session=session)

print("✅ Tavily search and MarkItDown ready")

---
## 🛠️ Part 1: The Two Tools

Both tools **fail soft**: a bad location, a timeout or a page that refuses to be scraped becomes a
message the agent can read and work around, not an exception that ends the run. The search tool
truncates each page to 1,500 characters so three pages cannot overflow the agent's context.

In [ ]:
# ============================================================================
# TRAVEL TOOLS: Weather (OpenWeatherMap) and web search with page extraction
# ============================================================================
from concurrent.futures import ThreadPoolExecutor, TimeoutError

from langchain_core.tools import tool

WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")
MAX_PAGE_CHARS = 1500


@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location (city name, e.g. 'Kolkata').

    Use this whenever the user asks about a destination, the weather, or the temperature.
    """
    print(f"🌤️ get_weather({location!r})")
    try:
        response = requests.get(
            "https://api.openweathermap.org/data/2.5/weather",
            params={"q": location, "appid": WEATHER_API_KEY, "units": "metric"},
            timeout=10,
        )
        data = response.json()
    except requests.exceptions.RequestException as e:
        return f"Error fetching weather data: {e}"

    if str(data.get("cod")) != "200":  # e.g. invalid key or unknown city: pass OpenWeatherMap's reason on
        return f"Weather API error for {location!r}: {data.get('message', data)}"
    main, wind = data["main"], data.get("wind", {})
    return (
        f"Weather for {data['name']}, {data.get('sys', {}).get('country', '')}:\n"
        f"- Temperature: {main['temp']}°C, feels like {main['feels_like']}°C\n"
        f"- Condition: {data['weather'][0]['description']}\n"
        f"- Humidity: {main['humidity']}%\n"
        f"- Wind: {round(wind.get('speed', 0) * 3.6, 1)} kph"
    )


def extract_content(url: str) -> str:
    """Convert one web page to text, truncated to MAX_PAGE_CHARS."""
    # Fetch with our own timeout (MarkItDown's own fetch has none), then convert the response
    page = md_converter.convert_response(session.get(url, timeout=10))
    title = (page.title or "No title").strip()
    text = (page.text_content or "").strip()
    if len(text) > MAX_PAGE_CHARS:
        text = text[:MAX_PAGE_CHARS] + "... [Content truncated]"
    return f"{title}\n{text}"


@tool
def search_web_extract_info(query: str) -> list:
    """Search the web and return the text of the top pages.

    Use this to find local attractions, things to do, travel guides, or any specific
    information you do not know.
    """
    print(f"🔍 search_web_extract_info({query!r})")
    results = tavily_search.invoke({"query": query})["results"]
    docs = []
    # Fetch pages in parallel; give each at most 15 seconds
    executor = ThreadPoolExecutor()
    futures = {r["url"]: executor.submit(extract_content, r["url"]) for r in results}
    for url, future in futures.items():
        try:
            docs.append(future.result(timeout=15))
        except TimeoutError:
            docs.append(f"Timeout extracting from: {url}")
        except Exception as e:
            docs.append(f"Error extracting from {url}: {e.__class__.__name__}")
    # Don't wait for stragglers: a slow page must not stall the agent
    executor.shutdown(wait=False, cancel_futures=True)
    return docs


tools = [search_web_extract_info, get_weather]
print(f"✅ Defined {len(tools)} tools: {[t.name for t in tools]}")

In [ ]:
# ============================================================================
# TEST: Each tool on its own
# ============================================================================
print(get_weather.invoke("Kolkata"))
print()
pages = search_web_extract_info.invoke("top attractions in Kolkata")
print(f"\n{len(pages)} page(s); first one starts:\n{pages[0][:300]}")

---
## 🎯 Part 2: Does the Model Pick Both Tools?

Before building the agent, check tool selection on its own: bind the tools and ask a question
that needs both. The model should return two tool calls (it does not run them).

In [ ]:
# ============================================================================
# TEST: Tool selection with bind_tools
# ============================================================================
response = llm.bind_tools(tools).invoke(
    "Show me Local Attractions in Bangalore and The Weather in Bangalore"
)
print("Tool calls identified by the LLM:")
for tc in response.tool_calls:
    print(f"  • {tc['name']}: {tc['args']}")

---
## 🤖 Part 3: The Agent

The system prompt states the use case (always attractions *and* weather) and the output shape.
It does **not** ask for `Thought / Action / Observation` text: with native tool calling the loop
is driven by structured tool calls, and that format would only leak into the final answer (see
`02_tool_calling_vs_react.ipynb`).

In [ ]:
# ============================================================================
# THE TRAVEL ASSISTANT: System prompt + create_agent + a call limit
# ============================================================================
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware

SYSTEM_PROMPT = """You are a helpful travel assistant.
When the user mentions a destination or location:
1. Search for local attractions and tourist spots with search_web_extract_info
2. Get the current weather for that location with get_weather
3. Answer with the top attractions as bullet points, followed by a short weather summary

Keep responses concise and well-formatted."""

travel_agent = create_agent(
    llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    middleware=[ModelCallLimitMiddleware(run_limit=6, exit_behavior="end")],
)

print("✅ Travel assistant agent created!")

---
## 🧪 Part 4: Test the Travel Assistant

In [ ]:
# ============================================================================
# RUN: A one-word destination query
# ============================================================================
from IPython.display import Markdown, display

query = "Kolkata"
print(f"🗺️ Query: {query}\n")
result = travel_agent.invoke({"messages": [("user", query)]})

used = [tc["name"] for m in result["messages"] for tc in getattr(m, "tool_calls", [])]
print(f"\n🔧 Tools called: {used}")
print(f"📞 Model calls: {sum(1 for m in result['messages'] if m.type == 'ai')}")
display(Markdown(result["messages"][-1].content))

### How the agent reasons

The run above is the ReAct loop with native tool calls:

1. **Problem analysis** — the user wrote only "Kolkata". The system prompt tells the model what a destination query needs: attractions and weather.
2. **Tool selection** — usually both tools in one turn (parallel calls), since neither depends on the other.
3. **Observation** — the page texts and the weather summary come back as `ToolMessage`s. Failed pages arrive as short error strings, so one bad site does not sink the answer.
4. **Iteration decision** — if a result is thin, the model may search again; `ModelCallLimitMiddleware(run_limit=6)` bounds that.
5. **Synthesis** — the final model call turns the observations into bullet points and a weather line.

| Design choice | Why |
|---|---|
| Truncate pages to 1,500 characters | Three full pages can exceed the context window; the attractions are near the top |
| Fail-soft tools | An exception ends the run; an error string lets the agent carry on |
| Model-call limit | Bounds cost if the agent keeps searching |
| No `Thought:` format in the prompt | Native tool calling needs none; it would leak into the answer |

---
## 📝 Summary

In this solution:

### 1. Two Robust Tools
- OpenWeatherMap and Tavily + MarkItDown, each returning readable text or a readable error

### 2. Checking Tool Selection
- `bind_tools` + one question shows whether the docstrings lead the model to the right tools

### 3. The Agent
- A use-case system prompt plus `create_agent` and a call limit is the whole agent

### 4. Reading the Run
- The tool calls in the message list show what the agent did; the last message is what the user sees

### Next Steps
- Try the bonus exercises in `modules/03_react_agents/exercises/01_intelligent_travel_assistant_agent.ipynb`
- **`01_model_integration.ipynb`** (module 04): models, messages, structured output and middleware around `create_agent`, in depth